# Session 05 - Feature Engineering Fundamentals

This notebook engineers features for a synthetic workplace churn dataset and compares a baseline feature set against an engineered feature set.

**Learning outcome:** design features that improve model learnability while preserving evaluation integrity.

**Professional rule:** split before model fitting, exclude future/target-derived columns, and document why each engineered feature is safe.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 1. Load the dataset
The dataset contains safe pre-prediction fields plus intentionally leaky post-outcome fields for exclusion practice.

In [2]:
DATA_PATH = 'workplace_churn_feature_engineering_dataset.csv'
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(720, 21)


,customer_id,signup_date,last_login_date,region,sector,plan_type,contract_type,acquisition_channel,monthly_fee_gbp,tenure_months,...,active_days_30d,support_tickets_90d,invoices_late_6m,satisfaction_score,renewal_month,account_notes,cancellation_reason,account_closed_date,retention_offer_after_churn,churned
0,CUST-1001,2022-09-21,2024-12-29,Midlands,Education,Basic,Monthly,Sales-led,55.75,27,...,5,1,0,4.9,4,Healthy adoption in team; users asked for repo...,NaN,NaN,NaN,0
1,CUST-1002,2021-02-22,2024-12-20,North West,Healthcare,Basic,Monthly,Sales-led,49.21,46,...,9,3,0,6.8,3,Invoice query raised; finance contact requeste...,NaN,NaN,NaN,0
2,CUST-1003,2021-01-21,2024-12-31,Northern Ireland,Professional Services,Basic,Monthly,Self-serve,36.68,47,...,8,1,4,7.1,7,Multiple support contacts; wants faster respon...,business_closed,2025-02-01,none,1
3,CUST-1004,2020-07-08,2024-12-10,Wales,Technology,Basic,Monthly,Sales-led,43.12,53,...,0,2,1,7.2,3,Support issue around onboarding and admin setup.,support,2025-02-07,success_call,1
4,CUST-1005,2020-01-07,2024-12-24,London,Professional Services,Basic,Annual,Self-serve,34.46,59,...,12,2,0,7.7,2,Healthy adoption in team; users asked for repo...,NaN,NaN,NaN,0


## 2. Inspect target balance and columns
Before engineering features, understand the target and identify columns that must not be used as inputs.

In [3]:
target = 'churned'
print(df[target].value_counts(normalize=True).rename('target_rate'))
print('\nColumns:')
print(df.columns.tolist())

churned
0    0.747222
1    0.252778
Name: target_rate, dtype: float64

Columns:
['customer_id', 'signup_date', 'last_login_date', 'region', 'sector', 'plan_type', 'contract_type', 'acquisition_channel', 'monthly_fee_gbp', 'tenure_months', 'usage_minutes_30d', 'active_days_30d', 'support_tickets_90d', 'invoices_late_6m', 'satisfaction_score', 'renewal_month', 'account_notes', 'cancellation_reason', 'account_closed_date', 'retention_offer_after_churn', 'churned']


## 3. Define leaky columns and safe baseline features
Leaky columns are not available at the prediction moment or encode the answer after churn has happened.

In [4]:
leaky_columns = ['cancellation_reason', 'account_closed_date', 'retention_offer_after_churn']
identifier_columns = ['customer_id']

baseline_numeric = [
    'monthly_fee_gbp', 'tenure_months', 'usage_minutes_30d', 'active_days_30d',
    'support_tickets_90d', 'invoices_late_6m', 'satisfaction_score', 'renewal_month'
]
baseline_categorical = ['region', 'sector', 'plan_type', 'contract_type', 'acquisition_channel']

X_base = df[baseline_numeric + baseline_categorical].copy()
y = df[target].copy()

print('Excluded as leaky:', leaky_columns)
print('Baseline columns:', X_base.columns.tolist())

Excluded as leaky: ['cancellation_reason', 'account_closed_date', 'retention_offer_after_churn']
Baseline columns: ['monthly_fee_gbp', 'tenure_months', 'usage_minutes_30d', 'active_days_30d', 'support_tickets_90d', 'invoices_late_6m', 'satisfaction_score', 'renewal_month', 'region', 'sector', 'plan_type', 'contract_type', 'acquisition_channel']


## 4. Create the same train/test split for all comparisons
This keeps the comparison fair. Only the feature set changes.

In [5]:
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train_base.shape, X_test_base.shape)

(540, 13) (180, 13)


## 5. Build a reusable modelling function
The model is intentionally simple. The session is about feature engineering, not advanced model selection.

In [6]:
def build_model(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_pipe, numeric_cols),
            ('cat', categorical_pipe, categorical_cols)
        ]
    )
    model = LogisticRegression(max_iter=1000)
    return Pipeline(steps=[('preprocess', preprocessor), ('model', model)])

def evaluate_pipeline(name, pipe, X_train, X_test, y_train, y_test):
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    return {
        'input_set': name,
        'accuracy': round(accuracy_score(y_test, pred), 3),
        'precision': round(precision_score(y_test, pred, zero_division=0), 3),
        'recall': round(recall_score(y_test, pred, zero_division=0), 3),
        'f1': round(f1_score(y_test, pred, zero_division=0), 3),
        'roc_auc': round(roc_auc_score(y_test, proba), 3),
        'n_features_before_encoding': X_train.shape[1]
    }

## 6. Baseline model
Train a simple model using only raw safe input columns.

In [7]:
baseline_pipe = build_model(baseline_numeric, baseline_categorical)
baseline_result = evaluate_pipeline('baseline_safe_raw_features', baseline_pipe, X_train_base, X_test_base, y_train, y_test)
baseline_result

{'input_set': 'baseline_safe_raw_features',
 'accuracy': 0.794,
 'precision': 0.618,
 'recall': 0.467,
 'f1': 0.532,
 'roc_auc': np.float64(0.82),
 'n_features_before_encoding': 13}

## 7. Engineer features with Pandas
Feature engineering converts raw fields into more learnable signals. Each feature must be checked for leakage.

In [8]:
def add_engineered_features(data):
    out = data.copy()
    # Date features - derived from dates known before prediction.
    signup = pd.to_datetime(out['signup_date'], errors='coerce')
    last_login = pd.to_datetime(out['last_login_date'], errors='coerce')
    prediction_date = pd.to_datetime('2024-12-31')
    out['signup_month'] = signup.dt.month
    out['signup_quarter'] = signup.dt.quarter
    out['days_since_last_login'] = (prediction_date - last_login).dt.days

    # Binning - convert continuous tenure into lifecycle groups.
    out['tenure_band'] = pd.cut(
        out['tenure_months'],
        bins=[0, 6, 18, 36, 999],
        labels=['new', 'growing', 'established', 'long_term']
    ).astype('object')

    # Transformations and ratios.
    out['usage_per_active_day'] = out['usage_minutes_30d'] / out['active_days_30d'].replace(0, np.nan)
    out['fee_per_usage_minute'] = out['monthly_fee_gbp'] / out['usage_minutes_30d'].replace(0, np.nan)
    out['ticket_rate_per_tenure'] = out['support_tickets_90d'] / out['tenure_months'].replace(0, np.nan)

    # Interaction-style features.
    out['low_usage_high_tickets_flag'] = ((out['usage_minutes_30d'] < 120) & (out['support_tickets_90d'] >= 3)).astype(int)
    out['monthly_contract_low_satisfaction_flag'] = ((out['contract_type'] == 'Monthly') & (out['satisfaction_score'] < 6)).astype(int)

    # Text-derived features from account notes.
    notes = out['account_notes'].fillna('').str.lower()
    out['account_note_length'] = notes.str.len()
    out['mentions_price_issue'] = notes.str.contains('price|invoice|fee|discount', regex=True).astype(int)
    out['mentions_support_issue'] = notes.str.contains('support|onboarding|response', regex=True).astype(int)
    return out

df_eng = add_engineered_features(df)
df_eng.filter(regex='tenure_band|usage_per|fee_per|ticket_rate|flag|note|mentions|signup_|days_since').head()

,signup_date,account_notes,signup_month,signup_quarter,days_since_last_login,tenure_band,usage_per_active_day,fee_per_usage_minute,ticket_rate_per_tenure,low_usage_high_tickets_flag,monthly_contract_low_satisfaction_flag,account_note_length,mentions_price_issue,mentions_support_issue
0,2022-09-21,Healthy adoption in team; users asked for repo...,9,3,2,established,22.400000,0.497768,0.037037,0,1,65,0,0
1,2021-02-22,Invoice query raised; finance contact requeste...,2,1,11,long_term,32.633333,0.167552,0.065217,0,0,62,1,0
2,2021-01-21,Multiple support contacts; wants faster respon...,1,1,0,long_term,15.350000,0.298697,0.021277,0,0,54,0,1
3,2020-07-08,Support issue around onboarding and admin setup.,7,3,21,long_term,NaN,0.249682,0.037736,0,0,48,0,1
4,2020-01-07,Healthy adoption in team; users asked for repo...,1,1,7,long_term,21.150000,0.135776,0.033898,0,0,65,0,0


## 8. Compare baseline vs engineered feature set
Use the same target, same test size, same random state, same model type, and same metric set.

In [9]:
engineered_numeric = baseline_numeric + [
    'signup_month', 'signup_quarter', 'days_since_last_login',
    'usage_per_active_day', 'fee_per_usage_minute', 'ticket_rate_per_tenure',
    'low_usage_high_tickets_flag', 'monthly_contract_low_satisfaction_flag',
    'account_note_length', 'mentions_price_issue', 'mentions_support_issue'
]
engineered_categorical = baseline_categorical + ['tenure_band']

X_eng = df_eng[engineered_numeric + engineered_categorical].copy()
X_train_eng, X_test_eng, _, _ = train_test_split(
    X_eng, y, test_size=0.25, random_state=42, stratify=y
)

engineered_pipe = build_model(engineered_numeric, engineered_categorical)
engineered_result = evaluate_pipeline('engineered_features', engineered_pipe, X_train_eng, X_test_eng, y_train, y_test)

results = pd.DataFrame([baseline_result, engineered_result])
results

,input_set,accuracy,precision,recall,f1,roc_auc,n_features_before_encoding
0,baseline_safe_raw_features,0.794,0.618,0.467,0.532,0.820,13
1,engineered_features,0.767,0.548,0.378,0.447,0.778,25


## 9. Document engineered features
A professional feature is not just code. It needs rationale, leakage status, and a caveat.

In [10]:
feature_summary = pd.DataFrame([
    {'feature':'tenure_band', 'type':'binning', 'rationale':'Captures customer lifecycle stage.', 'leakage_check':'Uses signup/tenure known before prediction.', 'caveat':'Bins may hide differences inside a band.'},
    {'feature':'usage_per_active_day', 'type':'ratio', 'rationale':'Measures usage intensity, not just total usage.', 'leakage_check':'Uses recent pre-prediction usage window.', 'caveat':'Missing or zero active days need careful handling.'},
    {'feature':'low_usage_high_tickets_flag', 'type':'interaction', 'rationale':'Combines weak adoption and support friction.', 'leakage_check':'Uses pre-prediction tickets and usage.', 'caveat':'May reflect service-quality issues, not customer intent.'},
    {'feature':'mentions_price_issue', 'type':'text-derived flag', 'rationale':'Captures price concern from account notes.', 'leakage_check':'Only safe if note exists before prediction.', 'caveat':'Manual notes can be inconsistent or biased.'},
])
feature_summary

,feature,type,rationale,leakage_check,caveat
0,tenure_band,binning,Captures customer lifecycle stage.,Uses signup/tenure known before prediction.,Bins may hide differences inside a band.
1,usage_per_active_day,ratio,"Measures usage intensity, not just total usage.",Uses recent pre-prediction usage window.,Missing or zero active days need careful handl...
2,low_usage_high_tickets_flag,interaction,Combines weak adoption and support friction.,Uses pre-prediction tickets and usage.,"May reflect service-quality issues, not custom..."
3,mentions_price_issue,text-derived flag,Captures price concern from account notes.,Only safe if note exists before prediction.,Manual notes can be inconsistent or biased.


## 10. Save outputs
These outputs can be placed in a GitHub repository with the notebook and report.

In [11]:
results.to_csv('baseline_vs_engineered_results.csv', index=False)
feature_summary.to_csv('engineered_feature_summary.csv', index=False)
df_eng.to_csv('workplace_churn_engineered_features.csv', index=False)
print('Saved results, feature summary, and engineered dataset.')

Saved results, feature summary, and engineered dataset.


## Student exercise
Create three additional engineered features. For each one, write:

1. Feature name.
2. Business rationale.
3. Why it may help a model.
4. Leakage check.
5. Possible bias or governance caveat.

### Student Exercise Answers

Three new features, each built from a baseline column Section 7 didn't already use on its own (`invoices_late_6m`, `renewal_month`, and a `satisfaction_score` / `monthly_fee_gbp` combination).

In [12]:
def add_student_engineered_features(data):
    out = data.copy()

    # 1. Billing risk flag - turns a skewed count into a simple threshold.
    out['frequent_late_payer_flag'] = (out['invoices_late_6m'] >= 2).astype(int)

    # 2. Renewal timing - cyclical months-to-renewal from the same fixed prediction date used in Section 7.
    prediction_date = pd.to_datetime('2024-12-31')
    out['months_to_renewal'] = (out['renewal_month'] - prediction_date.month) % 12

    # 3. Value-for-money ratio.
    out['satisfaction_per_fee'] = out['satisfaction_score'] / out['monthly_fee_gbp'].replace(0, np.nan)

    return out

df_student = add_student_engineered_features(df_eng)
df_student[['frequent_late_payer_flag', 'months_to_renewal', 'satisfaction_per_fee']].head()

,frequent_late_payer_flag,months_to_renewal,satisfaction_per_fee
0,0,4,0.087892
1,0,3,0.138183
2,1,7,0.193566
3,0,3,0.166976
4,0,2,0.223447


In [13]:
student_feature_summary = pd.DataFrame([
    {
        'feature': 'frequent_late_payer_flag',
        'business_rationale': 'Flags customers with 2 or more late invoices in the last 6 months, a concrete billing-friction signal account managers already watch for.',
        'why_it_may_help': 'Turns a raw, skewed count into a simple risk threshold the model can use directly, rather than relying on it to learn the right cut-off itself.',
        'leakage_check': 'invoices_late_6m only covers billing history up to the prediction point, so this is safe as long as the 6-month window ends before churn is decided.',
        'bias_or_governance_caveat': 'Billing friction can reflect financial hardship rather than dissatisfaction with the service - using this to target retention offers could disproportionately affect lower-income customers rather than genuinely at-risk ones.'
    },
    {
        'feature': 'months_to_renewal',
        'business_rationale': 'Customers are more likely to actively reconsider staying or leaving close to their renewal date, so proximity to renewal is a meaningful timing signal.',
        'why_it_may_help': 'Converts a calendar month into a cyclical distance-to-event feature, which is more directly meaningful to a churn model than the raw month number on its own.',
        'leakage_check': 'Safe only if renewal_month reflects the originally scheduled renewal date and is not updated after a retention conversation or cancellation - worth confirming with whoever maintains the field.',
        'bias_or_governance_caveat': 'If renewal timing correlates with contract type or sector (e.g. enterprise contracts renewing annually, individual plans monthly), this feature could act as a proxy for those groups rather than genuine churn risk.'
    },
    {
        'feature': 'satisfaction_per_fee',
        'business_rationale': 'Reflects perceived value for money - the same satisfaction score means something different for a customer paying a little each month versus a lot.',
        'why_it_may_help': 'Combines two individually weaker signals into a single ratio that may separate churn risk better than either variable alone.',
        'leakage_check': 'Inherits the safety of its two inputs; both are already used as baseline features and assumed available before the prediction point.',
        'bias_or_governance_caveat': 'The ratio is hard to interpret on its own and breaks down at low or zero monthly_fee_gbp; it could also indirectly encode plan_type, since fee level tracks plan choice closely.'
    }
])
student_feature_summary

,feature,business_rationale,why_it_may_help,leakage_check,bias_or_governance_caveat
0,frequent_late_payer_flag,Flags customers with 2 or more late invoices i...,"Turns a raw, skewed count into a simple risk t...",invoices_late_6m only covers billing history u...,Billing friction can reflect financial hardshi...
1,months_to_renewal,Customers are more likely to actively reconsid...,Converts a calendar month into a cyclical dist...,Safe only if renewal_month reflects the origin...,If renewal timing correlates with contract typ...
2,satisfaction_per_fee,Reflects perceived value for money - the same ...,Combines two individually weaker signals into ...,Inherits the safety of its two inputs; both ar...,The ratio is hard to interpret on its own and ...
